# 🔍 Phase 1 – Explainability: SHAP & LIME
**Goal:** Explain *why* a model makes predictions — critical for Finance & Healthcare.

Topics:
- Global feature importance with SHAP
- Local (per-prediction) SHAP explanations
- LIME for black-box models
- Waterfall, Beeswarm, Force plots

In [1]:
import shap
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

shap.initjs()
print('Libraries ready!')

ImportError: Numba needs NumPy 1.24 or less

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = GradientBoostingClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
print(f'Accuracy: {model.score(X_test, y_test):.4f}')

In [ ]:
# SHAP Global Importance
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
plt.sca(axes[0])
shap.plots.bar(shap_values, max_display=10, show=False, ax=axes[0])
axes[0].set_title('Global Feature Importance (SHAP)', fontweight='bold')
plt.sca(axes[1])
shap.plots.beeswarm(shap_values, max_display=10, show=False, ax=axes[1])
axes[1].set_title('SHAP Beeswarm: Impact Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/shap_global.png', dpi=150, bbox_inches='tight')
plt.show()
print('Blue=low feature value, Red=high feature value')

In [ ]:
# SHAP Local: single prediction
idx = 5
actual = 'Benign' if y_test[idx] == 1 else 'Malignant'
prob = model.predict_proba(X_test.iloc[[idx]])[0][1]
print(f'Patient #{idx}: Actual={actual}, P(Benign)={prob:.3f}')

shap.plots.waterfall(shap_values[idx], max_display=12)
plt.savefig('../data/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# LIME local explanation
lime_exp = lime.lime_tabular.LimeTabularExplainer(
    X_train.values, feature_names=X_train.columns.tolist(),
    class_names=['Malignant', 'Benign'], mode='classification', random_state=42
)
exp = lime_exp.explain_instance(X_test.iloc[idx].values, model.predict_proba, num_features=10)
fig = exp.as_pyplot_figure()
fig.set_size_inches(12, 6)
plt.title(f'LIME: Why patient #{idx} was predicted {"Benign" if prob>0.5 else "Malignant"}')
plt.tight_layout()
plt.savefig('../data/lime_explanation.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP vs LIME
| Aspect | SHAP | LIME |
|--------|------|------|
| Consistency | Theoretically grounded | Approximation |
| Speed | Fast (TreeSHAP) | Slower (perturbation) |
| Global view | Yes | Local only |
| Any model | KernelSHAP | Yes |

**Exercise:** Find a misclassified patient and use SHAP to diagnose why the model failed!